# Day 54 — Monitoring & model governance
Objectives:
- Monitor data quality and drift.
- Track model performance post-deployment.
- Governance basics: model registry, approvals, audit trail.
Note: This notebook simulates drift and basic monitoring metrics.

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(42)
# Simulate baseline score distribution
y_true = rng.integers(0,2, size=5000)
y_score = y_true*0.7 + (1-y_true)*0.3 + rng.normal(0,0.1, size=5000)
baseline_auc = roc_auc_score(y_true, y_score)
baseline_auc


In [ ]:
# Simulate drift: feature shift lowers separability
y_true2 = rng.integers(0,2, size=5000)
y_score2 = y_true2*0.6 + (1-y_true2)*0.4 + rng.normal(0,0.15, size=5000)
live_auc = roc_auc_score(y_true2, y_score2)
delta = live_auc - baseline_auc
baseline_auc, live_auc, delta


## Data drift detection (simple)
- Monitor feature means/stds, PSI (Population Stability Index) for scoring bands.
Below is a simple PSI demo for score bins.

In [ ]:
def psi(expected, actual, bins=10):
    e_perc, _ = np.histogram(expected, bins=bins, range=(expected.min(), expected.max()))
    a_perc, _ = np.histogram(actual,   bins=bins, range=(expected.min(), expected.max()))
    e_perc = e_perc / e_perc.sum(); a_perc = a_perc / a_perc.sum()
    # add tiny value to avoid div/0
    e_perc = np.clip(e_perc, 1e-6, None); a_perc = np.clip(a_perc, 1e-6, None)
    return np.sum((a_perc - e_perc) * np.log(a_perc / e_perc))
psi_value = psi(y_score, y_score2)
psi_value


## Governance checklist
- Register models in a Model Registry (MLflow, SageMaker, etc.).
- Record: training data snapshot/hash, code version (git sha), hyperparams, metrics.
- Approval workflow for promotion to production (staging → prod).
- Periodic re-validation; drift + performance alerts.

## Learner exercises and progressive hints

1. Compute PSI for multiple features or score windows over time.
2. Build a small pandas/Matplotlib dashboard of weekly AUC and PSI.
3. Draft a governance policy covering roles, approvals, alerts, and rollback.

### Progressive hints

1. Freeze each feature's reference bin edges and record sample counts beside PSI.
   Test “no shift,” mean shift, and variance shift.
2. Use one row per week with observation count, label coverage, AUC, PSI, and
   model version. Mark missing/delayed labels instead of filling fake metrics.
3. For every threshold, name an owner, review clock, evidence source, action,
   escalation path, and recovery test.

### Additional mastery practice

Design monitoring around observable data, delayed truth, action thresholds, ownership, and rehearsed recovery—not dashboards that merely display numbers.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Label-delay analysis:** Simulate labels arriving 14–30 days after predictions. Build separate views for immediate input/score health and matured performance cohorts.
   **Progressive hint:** Join outcomes by stable prediction ID and evaluate only cohorts whose label window has matured; report label coverage and censoring.
5. **Alert hysteresis:** Design warning and critical thresholds that require persistence or multiple windows, then show how hysteresis prevents alert flapping.
   **Progressive hint:** Use different enter and clear conditions, minimum support, and a cooldown. Preserve raw measurements for audit.
6. **Rollback drill:** Write and rehearse a rollback from model version B to A, including trigger, authority, artifact verification, traffic switch, smoke test, communication, and post-incident evidence.
   **Progressive hint:** A rollback is complete only when the prior artifact, schema, and dependencies remain loadable and the recovery check passes.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.


In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Label-delay analysis


# Practice 5 — Alert hysteresis


# Practice 6 — Rollback drill
